# Backend lazy proxies: run igraph & NetworkX algorithms on a rich AnnNet

AnnNet is the source of truth. igraph and NetworkX are *algorithm libraries* you
borrow on demand. The `G.ig` and `G.nx` namespaces are **lazy proxies**: they hold
no backend graph until you actually call an algorithm, then they convert once and
cache the result (keyed on the graph's version). Mutate the AnnNet and the cached
backend is transparently rebuilt.

This notebook uses a **real** network — the DoRothEA transcription-factor →
target-gene regulatory graph (~5000 genes, ~15000 directed interactions), fetched
live from OmniPath so the notebook runs anywhere with no data files. We drive igraph
and NetworkX algorithms straight off the AnnNet, by gene symbol, without ever holding
a backend object ourselves.

## 1. Build a rich AnnNet

We fetch the DoRothEA regulatory network **directly from the OmniPath web service** with `from_omnipath` — no data files to download by hand. It needs the `omnipath` package (`pip install omnipath`) and one network round-trip; OmniPath caches the response locally, so re-runs are fast. The result is a directed, signed graph with real edge annotations (`is_stimulation`, `is_inhibition`).

> `query={'genesymbols': True}` asks OmniPath for gene-symbol columns; without it the service returns UniProt accessions instead.

In [1]:
import warnings
from annnet.io.omnipath import from_omnipath

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    G = from_omnipath(
        dataset='dorothea',                 # fetched live from omnipathdb.org
        query={'genesymbols': True},        # return gene symbols, not UniProt IDs
        source_col='source_genesymbol',
        target_col='target_genesymbol',
        directed_col='is_directed',
        # Only pull real annotations — 'source'/'target' are reserved structural keys.
        edge_attr_cols=['is_stimulation', 'is_inhibition'],
        load_vertex_annotations=False,      # skip the ~114MB annotation archive download
    )

print('vertices :', G.nv)
print('edges    :', G.ne)
print('directed :', G.directed)
print('edge cols:', list(G.edge_attributes.columns))

[timing] fetch/receive df:     2.414s
[timing] column resolution:    0.0063s
         source='source_genesymbol'  target='target_genesymbol'  directed='is_directed'
         edge_attr_cols (2): ['is_stimulation', 'is_inhibition']
[timing] AnnNet init:          0.080s  (pre-sized n=15267 e=15267)
[timing] to_rows setup:        0.000s  (15267 rows, streaming=True)
[timing] bulk list build:      0.066s  (15267 edges)


[timing] add_edges_bulk:       0.099s
         vertices=5252  edges=15267
vertices : 5252
edges    : 15267
directed : True
edge cols: ['edge_id', 'is_stimulation', 'is_inhibition']


## 2. The proxy is lazy

`G.ig` / `G.nx` are just namespaces. Nothing is converted until you ask for a backend graph or call an algorithm. `.backend()` materialises (and caches) the converted graph; calling it again hands back the *same* object.

In [2]:
ig_graph = G.ig.backend()          # first call: convert AnnNet -> igraph
ig_graph_again = G.ig.backend()    # second call: served from cache

print(type(ig_graph).__module__, type(ig_graph).__name__)
print('vcount / ecount    :', ig_graph.vcount(), ig_graph.ecount())
print('same cached object :', ig_graph is ig_graph_again)

nx_graph = G.nx.backend()
print(type(nx_graph).__name__, '->', nx_graph.number_of_nodes(), 'nodes,',
      nx_graph.number_of_edges(), 'edges')

igraph Graph
vcount / ecount    : 5252 18266
same cached object : True


MultiDiGraph -> 5252 nodes, 18266 edges


## 3. igraph algorithms, driven from the AnnNet

Any igraph `Graph` method or top-level `igraph` function is reachable as `G.ig.<name>(...)`. The proxy substitutes the cached backend graph for you and maps results back to AnnNet vertex IDs (here: gene symbols).

In [3]:
# Betweenness identifies regulatory bottlenecks.
bt = G.ig.betweenness()
names = G.ig.backend().vs['name']
top_bt = sorted(zip(names, bt), key=lambda x: -x[1])[:8]

print('Top betweenness hubs (master regulators):')
for gene, score in top_bt:
    print(f'  {gene:8s} {score:12.1f}')

Top betweenness hubs (master regulators):
  MYC         1355290.7
  SP1         1099562.5
  TP53        1085433.2
  E2F1        1049322.2
  STAT1        944328.9
  CTCF         787783.6
  ETS1         649352.4
  EGR1         627043.3


In [4]:
# Community detection wants an undirected view: pass _ig_directed=False and the
# proxy rebuilds/caches an undirected conversion under a distinct cache key.
comm = G.ig.community_multilevel(_ig_directed=False)
print('communities :', len(comm))
print('modularity  :', round(comm.modularity, 3))
print('largest 3   :', sorted((len(c) for c in comm), reverse=True)[:3])

communities : 23
modularity  : 0.411
largest 3   : [1020, 744, 706]


## 4. Reference vertices by gene symbol

Because our vertex IDs *are* gene symbols, we call algorithms with human-readable names directly — the proxy coerces them to the backend's integer vertex indices and maps outputs back. (When your labels differ from the IDs, pass `_ig_label_field=<column>` / `_nx_label_field=<column>` to point at the label column.)

In [5]:
# Shortest directed path length from MYC to the cell-cycle inhibitor CDKN1A.
d = G.ig.distances(source='MYC', target='CDKN1A', mode='out')
print('igraph MYC -> CDKN1A hops:', d[0][0])

igraph MYC -> CDKN1A hops: 1


## 5. The same graph, through NetworkX

`G.nx.<name>(...)` reaches any NetworkX algorithm. Pass `G` itself where the function expects a graph — the proxy swaps in the cached NetworkX conversion.

In [6]:
# Named shortest path — output vertices come back as gene symbols.
path = G.nx.shortest_path(G, source='MYC', target='CDKN1A')
print('NetworkX MYC -> CDKN1A path:', path)

# PageRank as a {gene: score} dict.
pr = G.nx.pagerank(G)
top_pr = sorted(pr.items(), key=lambda kv: -kv[1])[:5]
print('\nTop PageRank genes:')
for gene, score in top_pr:
    print(f'  {gene:8s} {score:.5f}')

NetworkX MYC -> CDKN1A path: ['MYC', 'CDKN1A']

Top PageRank genes:
  SP1      0.01612
  MYC      0.01342
  E2F1     0.01025
  TP53     0.01000
  NFKB1    0.00884


## 6. Mutating the AnnNet invalidates the cached backends

The cache is keyed on the graph's internal version counter. Any structural edit bumps the version, so the next proxy call reconverts — you never hand-manage staleness.

In [7]:
before = G.ig.backend()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    G.add_vertices(['__PROBE__'])   # structural mutation -> version bump

after = G.ig.backend()

print('same object after mutation :', before is after)     # False: rebuilt
print('vcount before / after      :', before.vcount(), '/', after.vcount())

same object after mutation : False
vcount before / after      : 5252 / 5253


## 7. Two caveats worth knowing

**Lossy conversions warn.** Hyperedges and multiple layers/slices don't exist in
plain igraph/NetworkX. When you convert a graph that has them, the proxy emits a
`RuntimeWarning` describing what was dropped or flattened — the AnnNet itself is
untouched, but the borrowed backend view is a projection.

**Scalar integer outputs get vertex-mapped.** The proxy maps integer results back to
vertex IDs so that node-valued outputs (paths, component members) read naturally.
That means a function whose result is a plain *count* (e.g.
`number_weakly_connected_components`) will have its integer mapped to whatever vertex
sits at that row index — surprising. Prefer the collection-returning variant and take
`len(...)` yourself.

In [8]:
# DON'T rely on scalar-int returns through the proxy:
mapped = G.nx.number_weakly_connected_components(G)
print('number_weakly_connected_components ->', repr(mapped), '  # <- int mapped to a vertex id!')

# DO take the collection and measure it:
n_wcc = len(list(G.nx.weakly_connected_components(G)))
print('len(weakly_connected_components)   ->', n_wcc, '  # correct')

number_weakly_connected_components -> 'STAT5A'   # <- int mapped to a vertex id!
len(weakly_connected_components)   -> 10   # correct


## Recap

- `G.ig` / `G.nx` are **lazy** — no backend graph exists until an algorithm is called.
- The conversion is **cached per graph version** and reused across algorithm calls;
  mutating the AnnNet transparently invalidates it.
- Vertices are addressed by their AnnNet IDs (or a label column via `_ig_label_field` /
  `_nx_label_field`), and results are mapped back to those IDs.
- The AnnNet stays the single source of truth; igraph and NetworkX are disposable,
  on-demand computation backends.